# Pipeline de ingesta: AGETIC → NocoDB

**Ingeniería de Datos (IDT 601) · Bloque 2 · Sesión 3 — demo del docente**

Pipeline de punta a punta con el esqueleto de la sesión:

> **extraer → transformar → cargar → verificar**

- **Fuente (internet):** encuesta pública de AGETIC (CSV, 189 columnas).
- **Destino:** una tabla en **NocoDB Cloud** (grilla tipo hoja de cálculo).
- **Patrón:** carga completa **idempotente** (borra y recarga).

## Setup previo (una sola vez)

1. Entrá a <https://app.nocodb.com> (plan gratis) y creá una **base**.
2. Creá una **tabla** con estos 17 campos (tipo *Single line text* o *Number* para las numéricas):

   `Fue parte del proyecto`, `Sexo`, `Edad`, `Departamento`, `Ciudad`,
   `Años de experiencia`, `Tiempo en la unidad educativa`, `Años en la unidad educativa`,
   `Capacitación en TIC`, `Número de computadoras`, `Celular inteligente`,
   `Tipo de conexión`, `Lugar de acceso a internet`, `Sistema operativo`,
   `Autocalificación software libre (1–5)`, `Conoce licencias libres`, `Conoce datos abiertos`.

3. Generá un **API Token**: menú de cuenta → *Tokens* → *Create token*.
4. El **Table ID** está en la URL de la tabla (`.../nc/.../table/`**`tblXXXX`**) o vía API.

> Los valores van abajo, en la celda de configuración.


## 0. Configuración

Completá tus datos de NocoDB (en Colab podés usar *Secrets* en vez de pegarlos).

In [ ]:
# Configuración — reemplazá estos valores por los de TU instancia
NOCODB_URL       = "https://app.nocodb.com"      # o tu URL self-hosted
NOCODB_API_TOKEN = "PEGAR_TU_API_TOKEN"          # NocoDB → Account → Tokens
TABLE_ID         = "PEGAR_TABLE_ID"              # ej. "tblXXXXXXXXXXXX"
CSV_URL          = "https://datos.gob.bo/dataset/bf97cf5a-a8d3-4694-926c-2a3da3df40e5/resource/119afffe-8e0a-4657-9cea-40fd6837b90c/download/profesores_fin.csv"


In [ ]:
import io
import requests
import pandas as pd

print("pandas", pd.__version__)


## 1. Extraer

Descargamos el CSV. Ojo: el servidor de AGETIC exige un **User-Agent**, así que usamos `requests` en lugar de `pd.read_csv(url)`.

In [ ]:
resp = requests.get(CSV_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
resp.raise_for_status()

crudo = pd.read_csv(io.BytesIO(resp.content), dtype=str, encoding="utf-8-sig")
print("Filas:", len(crudo), "| Columnas:", crudo.shape[1])
crudo.head(3)


## 2. Transformar

Nos quedamos con las 17 columnas, las renombramos, unificamos `NA`/`N/A` → vacío y casteamos las numéricas.

In [ ]:
RENOMBRE = {
    "a0": "Fue parte del proyecto",
    "a1": "Sexo",
    "a2": "Edad",
    "a3": "Departamento",
    "a4": "Ciudad",
    "b2": "Años de experiencia",
    "b3": "Tiempo en la unidad educativa",
    "b4": "Años en la unidad educativa",
    "b8": "Capacitación en TIC",
    "c3": "Número de computadoras",
    "d2": "Celular inteligente",
    "e7": "Tipo de conexión",
    "e8": "Lugar de acceso a internet",
    "g1": "Sistema operativo",
    "g21": "Autocalificación software libre (1–5)",
    "h1": "Conoce licencias libres",
    "h3": "Conoce datos abiertos",
}
NUMERICAS = ["Edad", "Años de experiencia", "Años en la unidad educativa",
             "Número de computadoras", "Autocalificación software libre (1–5)"]

limpio = crudo[list(RENOMBRE)].rename(columns=RENOMBRE).copy()
limpio = limpio.replace({"NA": "", "N/A": ""})
for c in limpio.columns:
    limpio[c] = limpio[c].fillna("").astype(str).str.strip()
for c in NUMERICAS:
    limpio[c] = pd.to_numeric(limpio[c].replace("", None), errors="coerce")

print(limpio.dtypes)
limpio.head(3)


## 3. Verificar (local, antes de cargar)

Mismos controles que el taller. La tabla esperada: **871 · 848 · 9 · 20/78 · 0 NA**.

In [ ]:
print("Filas:", len(limpio))
print("Fueron parte del proyecto (Sí):", int((limpio["Fue parte del proyecto"] == "Sí").sum()))
print("Departamentos distintos:", limpio["Departamento"].nunique())
print("Edad mínima / máxima:", limpio["Edad"].min(), "/", limpio["Edad"].max())
print("Celdas NA/N/A restantes:", int(limpio.isin(["NA", "N/A"]).sum().sum()))


## 4. Cargar en NocoDB (carga completa idempotente)

Borramos todas las filas de la tabla y reinsertamos. Repetir la carga produce el mismo resultado, sin duplicados.

In [ ]:
BASE = NOCODB_URL.rstrip("/")
API = f"{BASE}/api/v2/tables/{TABLE_ID}/records"
HEADERS = {"xc-token": NOCODB_API_TOKEN, "Content-Type": "application/json"}


def borrar_todo():
    """Borra todas las filas de la tabla (recorre los IDs y los elimina)."""
    r = requests.get(API, headers=HEADERS, params={"pageSize": 1000}, timeout=60)
    r.raise_for_status()
    ids = [{"Id": fila["Id"]} for fila in r.json().get("list", [])]
    if ids:
        d = requests.delete(API, headers=HEADERS, json=ids, timeout=60)
        d.raise_for_status()
    return len(ids)


def cargar(registros, lote=200):
    """Inserta los registros en lotes para no saturar la API."""
    total = 0
    for i in range(0, len(registros), lote):
        chunk = registros[i:i + lote]
        p = requests.post(API, headers=HEADERS, json=chunk, timeout=120)
        p.raise_for_status()
        total += len(chunk)
    return total


In [ ]:
registros = limpio.where(pd.notna(limpio), None).to_dict(orient="records")
print("Filas borradas:", borrar_todo())
print("Filas insertadas:", cargar(registros))


## 5. Verificar contra NocoDB

Leemos la tabla ya cargada y confirmamos los conteos: la prueba de que el pipeline funcionó de punta a punta.

In [ ]:
r = requests.get(API, headers=HEADERS, params={"pageSize": 1000}, timeout=60)
r.raise_for_status()
df_db = pd.DataFrame(r.json().get("list", []))

print("Filas en NocoDB:", len(df_db))
print("Fueron parte del proyecto (Sí):", int((df_db["Fue parte del proyecto"] == "Sí").sum()))
print("Departamentos distintos:", df_db["Departamento"].nunique())
df_db.head(3)


## Cierre

Corriste el pipeline completo en código. En la guía de la sesión está la misma demo **en n8n** (por nodos):
dos herramientas distintas, **un mismo pipeline**.

- **Extraer** → descargar la fuente.
- **Transformar** → 17 columnas, limpiar y tipar.
- **Cargar** → tabla NocoDB (carga completa idempotente).
- **Verificar** → contar y comparar con lo esperado.

> Idea para la clase: *la herramienta cambia, las etapas no.*
